# Sentinel-3 OLCI Water

* **Products used:** 
[s3_olci_l2_wfr](https://explorer.digitalearth.africa/products/s3_olci_l2_wfr)

## Background

The Sentinel-3 OLCI Level-2 Water Full Resolution (OL_2_WFR) product provides atmospherically corrected observations and biophysical parameters derived from the Ocean and Land Colour Instrument (OLCI) onboard Sentinel-3A and 3B satellites. The product delivers consistent measurements of water-leaving reflectance, water quality, and atmospheric properties at 300 m spatial resolution with an approximately daily revisit frequency when both Sentinel-3 satellites are combined, making it valuable for a wide range of inland water, coastal, and marine applications, including:

* Monitoring chlorophyll-a concentration and phytoplankton dynamics
* Detecting harmful algal blooms
* Assessing water clarity and light attenuation
* Monitoring suspended sediments and turbidity
* Mapping coloured dissolved organic matter (CDOM)
* Monitoring inland, coastal, and marine water quality
* Supporting aquatic ecosystem and environmental monitoring
* Supporting long-term climate and water resource studies

The algorithms and retrieval approaches have been developed by the European Space Agency (ESA) and partners, ensuring continuity with MERIS heritage and improved spectral and radiometric performance. Validation draws on ground-based measurements and inter-comparison with other satellite products. More information on the product can be found in the OLCI Water User Guide and Algorithm Theoretical Basis Documents ([ATBDs](https://sentiwiki.copernicus.eu/web/s3-documents)).
***

## Description

This notebook will cover following topics:

1. Inspecting the products and measurements available in the datacube
2. Loading Sentinel-3 OLCI Data.
3. Plotting the results

***

## Getting started

To run this analysis, run all the cells in the notebook, starting with the "Load packages" cell.

### Load packages
Import Python packages that are used for the analysis.

In [1]:
%matplotlib inline

import datacube
import pandas as pd
import geopandas as gpd
from odc.geo.geom import Geometry

from deafrica_tools.plotting import display_map
from deafrica_tools.areaofinterest import define_area

### Connect to the datacube

Connect to the datacube so we can access DE Africa data.

In [2]:
dc = datacube.Datacube(app="Sentinel_3")

### List products

We can use datacube's `list_products` functionality to inspect DE Africa's products that are available in the datacube. The table below shows the product names that we will use to load the data, a brief description of the data, and the satellite instrument that acquired the data.

In [3]:
dc.list_products().loc[dc.list_products()['description'].str.contains('Sentinel-3')]

,name,description,license,default_crs,default_resolution
name,,,,,
s3_ol_2_wfr_nrt,s3_ol_2_wfr_nrt,Sentinel-3 Level 2 Water Full Resolution (WFR)...,CC-BY-4.0,EPSG:4326,"Resolution(x=0.003, y=-0.003)"
s3_olci_l2_lfr,s3_olci_l2_lfr,Sentinel-3 OLCI L2 LAND,CC-BY-4.0,EPSG:4326,"Resolution(x=0.003, y=-0.003)"
s3_olci_l2_wfr,s3_olci_l2_wfr,Sentinel-3 OLCI L2 WATER,CC-BY-4.0,EPSG:4326,"Resolution(x=0.003, y=-0.003)"
s3_slstr_l2_lst,s3_slstr_l2_lst,Sentinel-3 LST,CC-BY-4.0,EPSG:4326,"Resolution(x=0.003, y=-0.003)"
s3_syn_2_vg1,s3_syn_2_vg1,Sentinel-3 Level-2 VG1 SYN product,CC-BY-4.0,EPSG:4326,"Resolution(x=0.01, y=-0.01)"


Please choose the product by specifying its name, with the product `s3_olci_l2_wfr`

In [4]:
product = "s3_olci_l2_wfr"

### List measurements

We can further inspect the data available for the Sentinel-3 Land `s3_olci_l2_wfr` product using datacube's `list_measurements` functionality. The table below lists each of the measurements available in the data.

In [5]:
measurements = dc.list_measurements()
measurements.loc[product]

,name,dtype,units,nodata,aliases,flags_definition,add_offset,scale_factor
measurement,,,,,,,,
A865,A865,int16,1,32767.0,[aerosol_angstrom_exponent],NaN,0.0,0.0001
ADG443_NN,ADG443_NN,int16,log10(1/m),32767.0,[CDM_absorbtion_coefficient],NaN,0.0,0.0001
B01,B01,uint16,1,65535.0,[Oa01_reflectance],NaN,0.0,0.0001
B02,B02,uint16,1,65535.0,[Oa02_reflectance],NaN,0.0,0.0001
B03,B03,uint16,1,65535.0,[Oa03_reflectance],NaN,0.0,0.0001
B04,B04,uint16,1,65535.0,[Oa04_reflectance],NaN,0.0,0.0001
B05,B05,uint16,1,65535.0,[Oa05_reflectance],NaN,0.0,0.0001
B06,B06,uint16,1,65535.0,[Oa06_reflectance],NaN,0.0,0.0001
B07,B07,uint16,1,65535.0,[Oa07_reflectance],NaN,0.0,0.0001


The Sentinel-3 water product has 26 measurements:

| Measurement | Description | Application |
|-------------|-------------|-------------|
| **A865** (Aerosol Ångström Exponent) | Aerosol parameter describing the size distribution of atmospheric particles. | Atmospheric correction, aerosol characterisation, and climate studies. |
| **ADG443_NN** (Coloured Dissolved and Detrital Matter Absorption Coefficient) | Absorption coefficient of coloured dissolved organic matter (CDOM) and detrital material at 443 nm, estimated using a neural network algorithm. | Monitoring dissolved organic matter, freshwater inputs, and water quality. |
| **B01** (Oa01) | Sensitive to aerosol correction and improved water constituent retrieval. | Atmospheric correction and water constituent retrieval. |
| **B02** (Oa02) | Sensitive to yellow substances, detrital pigments, and turbidity. | CDOM monitoring, turbidity assessment, and water quality analysis. |
| **B03** (Oa03) | Corresponds to the chlorophyll absorption maximum and supports biogeochemistry and vegetation studies. | Chlorophyll retrieval, ocean colour analysis, biogeochemistry, and vegetation monitoring. |
| **B04** (Oa04) | Sensitive to chlorophyll. | Chlorophyll retrieval and water quality monitoring. |
| **B05** (Oa05) | Sensitive to chlorophyll, sediment, turbidity, and red tide. | Chlorophyll retrieval, sediment monitoring, turbidity assessment, and harmful algal bloom detection. |
| **B06** (Oa06) | Chlorophyll reference (minimum) band. | Chlorophyll retrieval and water quality analysis. |
| **B07** (Oa07) | Sensitive to sediment loading. | Sediment transport and turbidity monitoring. |
| **B08** (Oa08) | Corresponds to the second chlorophyll absorption maximum and is sensitive to sediment, yellow substances, and vegetation. | Chlorophyll retrieval, sediment monitoring, CDOM analysis, and vegetation monitoring. |
| **B09** (Oa09) | Optimized for improved chlorophyll fluorescence retrieval. | Chlorophyll fluorescence retrieval and ocean colour analysis. |
| **B10** (Oa10) | Corresponds to the chlorophyll fluorescence peak and red edge. | Chlorophyll fluorescence monitoring and vegetation analysis. |
| **B11** (Oa11) | Represents the chlorophyll fluorescence baseline and red-edge transition. | Chlorophyll fluorescence monitoring and vegetation analysis. |
| **B12** (Oa12) | Sensitive to oxygen absorption, clouds, and vegetation. | Cloud detection, atmospheric correction, and vegetation monitoring. |
| **B16** (Oa16) | Used for atmospheric and aerosol correction. | Atmospheric correction and aerosol retrieval. |
| **B17** (Oa17) | Supports atmospheric and aerosol correction, cloud detection, and pixel co-registration. | Atmospheric correction, cloud detection, and image co-registration. |
| **B18** (Oa18) | Water vapour absorption reference band and common reference band with SLSTR for vegetation monitoring. | Water vapour correction and vegetation monitoring. |
| **B21** (Oa21) | Sensitive to water vapour absorption and atmospheric/aerosol correction. | Water vapour retrieval and atmospheric correction. |
| **CHL_NN** (Algal Pigment – Complex Waters) | Chlorophyll-*a* concentration estimated using a neural network algorithm for optically complex coastal and inland waters. | Water quality monitoring, harmful algal bloom detection, and ecosystem health assessment. |
| **CHL_OC4ME** (Algal Pigment – Open Waters) | Chlorophyll-*a* concentration estimated using the OC4ME algorithm for open ocean waters. | Ocean productivity studies, phytoplankton monitoring, and marine ecosystem assessment. |
| **IWV_W** (Integrated Water Vapour) | Column-integrated atmospheric water vapour above each observation. | Atmospheric correction, climate studies, and environmental monitoring. |
| **KD490_M07** (Diffuse Attenuation Coefficient at 490 nm) | Indicates how quickly light at 490 nm is attenuated within the water column, providing a measure of water clarity. | Water clarity assessment, aquatic habitat monitoring, and primary productivity studies. |
| **PAR** (Photosynthetically Active Radiation) | Amount of sunlight available for photosynthesis reaching the Earth's surface. | Marine primary productivity studies, phytoplankton growth, and ecosystem monitoring. |
| **T865** (Aerosol Optical Thickness) | Measure of the amount of aerosols in the atmosphere above each observation. | Atmospheric correction, air quality studies, and climate monitoring. |
| **TSM_NN** (Total Suspended Matter) | Concentration of suspended particles in the water estimated using a neural network algorithm. | Sediment transport studies, turbidity monitoring, and coastal and inland water management. |
| **dataMask** (Data Mask) | Binary mask indicating valid and invalid observations. | Excluding missing or invalid observations and ensuring reliable data analysis. |

### Analysis parameters

The following cell sets the parameters, which define the area of interest to conduct the analysis over.
#### Select location
To define the area of interest, there are two methods available:

1. By specifying the latitude, longitude, and buffer, or separate latitude and longitude buffers, this method allows you to define an area of interest around a central point. You can input the central latitude, central longitude, and a buffer value in degrees to create a square area around the center point. For example, `lat = 10.338`, `lon = -1.055`, and `buffer = 0.1` will select an area with a radius of 0.1 square degrees around the point with coordinates `(10.338, -1.055)`. 
    
    Alternatively, you can provide separate buffer values for latitude and longitude for a rectangular area. For example, `lat = 10.338`, `lon = -1.055`, and `lat_buffer = 0.1` and`lon_buffer = 0.08` will select a rectangular area extending 0.1 degrees north and south, and 0.08 degrees east and west from the point `(10.338, -1.055)`.

   For reasonable loading times, set the buffer as `0.1` or lower.

3. By uploading a polygon as a `GeoJSON or Esri Shapefile`. If you choose this option, you will need to upload the geojson or ESRI shapefile into the Sandbox using Upload Files button <img align="top" src="../Supplementary_data/upload_files_icon.png"> in the top left corner of the Jupyter Notebook interface. ESRI shapefiles must be uploaded with all the related files `(.cpg, .dbf, .shp, .shx)`. Once uploaded, you can use the shapefile or geojson to define the area of interest. Remember to update the code to call the file you have uploaded.

To use one of these methods, you can uncomment the relevant line of code and comment out the other one. To comment out a line, add the `"#"` symbol before the code you want to comment out. By default, the first option which defines the location using latitude, longitude, and buffer is being used.

**If running the notebook for the first time**, keep the default settings below.
f running the notebook for the first time, keep the default settings below. This will demonstrate how the analysis works and provide meaningful results. The example uses Lake Mweru, a large freshwater lake located on the border between the Democratic Republic of the Congo (DRC) and Zambia. Lake Mweru supports important fisheries and surrounding communities, making it well suited for demonstrating Sentinel-3 OLCI water quality products such as chlorophyll-a, suspended matter, and water clarity.

**To run the notebook for a different area**, make sure Sentinel-3 OLCI Land data is available for the chosen area using the [DEAfrica Explorer](https://explorer.digitalearth.africa).

In [6]:
# Method 1: Specify the latitude, longitude, and buffer)
aoi = define_area(lat= -8.9793, lon= 28.7650, buffer=0.55)

# Method 2: Use a polygon as a GeoJSON or Esri Shapefile. 
# aoi = define_area(vector_path='aoi.shp')

#Create a geopolygon and geodataframe of the area of interest
geopolygon = Geometry(aoi["features"][0]["geometry"], crs="epsg:4326")
geopolygon_gdf = gpd.GeoDataFrame(geometry=[geopolygon], crs=geopolygon.crs)

# Get the latitude and longitude range of the geopolygon
lat_range = (geopolygon_gdf.total_bounds[1], geopolygon_gdf.total_bounds[3])
lon_range = (geopolygon_gdf.total_bounds[0], geopolygon_gdf.total_bounds[2])

In [7]:
display_map(x=lon_range, y=lat_range)

## Load Sentinel-3 dataset using `dc.load()`

Now that we know what products and measurements are available for the product, we can load data from the datacube using `dc.load`. We will load data from spectral satellite bands. By specifying `output_crs='EPSG:4326'` and `resolution=(-0.003, -0.003`), we request that datacube reproject our data to the global geographic coordinate reference system (CRS), with 300 x 300 m pixels. Finally, `group_by='solar_day'` ensures that overlapping images taken within seconds of each other as the satellite passes over are combined into a single time step in the data.

In [ ]:
query = {
    'x': (lon_range),
    'y': (lat_range),
    'time':('2026-05-01', '2026-05-30'),
    'output_crs': 'EPSG:4326',
    'resolution': (-0.003, 0.003)}

ds_S3 = dc.load(product=product,
                group_by="solar_day",
                **query)

ds_S3

## Scale factors
Many measurements in the Sentinel-3 OLCI Level-2 Water Full Resolution product are stored as integer values to reduce file size while preserving precision. The physical value of a measurement is obtained by applying the **scale factor** and **add offset** provided in the dataset metadata.

$$
\text{Physical value} = (\text{Stored value} \times \text{Scale factor}) + \text{Add offset}
$$

For example, a stored reflectance value of **2449** for **B01** with a scale factor of **0.0001** corresponds to a reflectance of **0.2449**.

The scale factor varies between measurements. Most OLCI reflectance bands use a scale factor of **0.0001**, while **IWV_W** (Integrated Water Vapour) uses a scale factor of **1.0**, meaning its stored values are already expressed in physical units.

The following example displays the scale factor, add offset and units for all the measurements.

In [ ]:
metadata = pd.DataFrame(
    [
        {
            "Measurement": measurement,
            "Scale factor": ds_S3[measurement].attrs.get("scale_factor"),
            "Add offset": ds_S3[measurement].attrs.get("add_offset"),
            "Units": ds_S3[measurement].attrs.get("units"),
        }
        for measurement in ds_S3.data_vars
    ]
)

metadata

## Masking

Before visualisation, we use the `dataMask` band to mask values affected by cloud or other issues. The code below keeps data for pixels where the data mask value is 1.

In [ ]:
ds_S3 = ds_S3.where(ds_S3.dataMask == 1)

#### Visualising OLCI reflectance bands

The cell below visualizes the `Band 03 (B03)` surface reflectance at the time of observation, representing atmospherically corrected reflectance in the blue-green region of the electromagnetic spectrum. The scale factor is applied prior to visualization to convert the stored integer values to physical reflectance values.

In [ ]:
(ds_S3['B03'].isel(time=[0, 8, 18, 24])*ds_S3["B03"].attrs["scale_factor"]).plot(robust=True, col="time", col_wrap=4);

#### Visualise Sentinel-3 OLCI Water `CHL_NN` band
The cell below visualizes the CHL_NN chlorophyll-a product, estimated using a neural network algorithm for optically complex coastal and inland waters. The scale factor is applied prior to visualization to convert the stored integer values to physical values.

In [ ]:
(ds_S3.isel(time=[0, 8, 18, 24])['CHL_NN']*ds_S3["CHL_NN"].attrs["scale_factor"]).plot(robust=True, col="time", col_wrap=4)

#### Visualise Sentinel-3 OLCI Water `CHL_OC4ME` band
The cell below visualizes the `CHL_OC4ME chlorophyll-a product`, estimated using the OC4ME algorithm for open ocean waters. The scale factor is applied prior to visualization to convert the stored integer values to physical values

In [ ]:
(ds_S3.isel(time=[0, 8, 18, 24])['CHL_OC4ME']*ds_S3["CHL_OC4ME"].attrs["scale_factor"]).plot(robust=True, col="time", col_wrap=4)

#### Visualise Sentinel-3 OLCI Water `TSM_NN` band
The cell below visualizes the `Total Suspended Matter (TSM_NN) product`, representing the concentration of suspended particles within the water column estimated using a neural network algorithm. The scale factor is applied prior to visualization to convert the stored integer values to physical values.

In [ ]:
(ds_S3.isel(time=[0, 8, 18, 24])['TSM_NN']*ds_S3["TSM_NN"].attrs["scale_factor"]).plot(robust=True, col="time", col_wrap=4)

#### Visualise Sentinel-3 OLCI Water OLCI KD490_M07 band
The cell below visualizes the `Diffuse Attenuation Coefficient at 490 nm (KD490_M07)`, representing water clarity and the rate at which light is attenuated through the water column. The scale factor is applied prior to visualization to convert the stored integer values to physical values.

In [ ]:
(ds_S3.isel(time=[0, 8, 18, 24])['KD490_M07']*ds_S3["KD490_M07"].attrs["scale_factor"]).plot(robust=True, col="time", col_wrap=4)

#### Visualise Sentinel-3 OLCI Water OLCI PAR band
The cell below visualizes `Photosynthetically Active Radiation (PAR)`, representing the amount of sunlight available for photosynthesis reaching the Earth's surface. As `PAR` has a scale factor of `1.0`, no scaling is required prior to visualization.

In [ ]:
(ds_S3.isel(time=[0, 8, 18, 24])['PAR']).plot(robust=True, col="time", col_wrap=4)

#### Visualise Sentinel-3 OLCI Water T865 band
The cell below visualizes the `Aerosol Optical Thickness (T865)`, representing the amount of aerosols in the atmosphere above each observation. The scale factor is applied prior to visualization to convert the stored integer values to physical values.

In [ ]:
(ds_S3.isel(time=[0, 8, 18, 24])['T865']*ds_S3["T865"].attrs["scale_factor"]).plot(robust=True, col="time", col_wrap=4)

#### Visualise Sentinel-3 OLCI Water OLCI IWV_W band
The cell below visualizes `Integrated Water Vapour (IWV_W)`, representing the total atmospheric water vapour contained within a vertical column of the atmosphere. As `IWV_W` has a scale factor of `1.0`, no scaling is required prior to visualization.

In [ ]:
(ds_S3.isel(time=[0, 8, 18, 24])['IWV_W']*ds_S3["IWV_W"]).plot(robust=True, col="time", col_wrap=4)

---

## Additional information

<b> License </b> The code in this notebook is licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0).

Digital Earth Africa data is licensed under the [Creative Commons by Attribution 4.0](https://creativecommons.org/licenses/by/4.0/) license.

<b> Contact </b> If you need assistance, please post a question on the [DE Africa Slack channel](https://digitalearthafrica.slack.com/) or on the [GIS Stack Exchange](https://gis.stackexchange.com/questions/ask?tags=open-data-cube) using the `open-data-cube` tag (you can view previously asked questions [here](https://gis.stackexchange.com/questions/tagged/open-data-cube)).

If you would like to report an issue with this notebook, you can file one on [Github](https://github.com/digitalearthafrica/deafrica-sandbox-notebooks).

<b> Compatible datacube version </b>

In [ ]:
print(datacube.__version__)

**Last Tested:**

In [ ]:
from datetime import datetime
datetime.today().strftime('%Y-%m-%d')